# Sohn et al. (2019) on DANDI:000130: single-trial GPFA

Paper: Sohn H, Narain D, Meirhaeghe N, Jazayeri M (2019). Bayesian computation through cortical
latent dynamics. *Neuron* 103, 934-947. doi:10.1016/j.neuron.2019.06.012.

Dataset: [DANDI:000130](https://dandiarchive.org/dandiset/000130/0.241017.1448), version
0.241017.1448, the version asked for in the prompt of this attempt and the dandiset's published
version. Claims: `claims/sohn-2019.yaml`, written from the accepted author manuscript
(PMC6805134).

This notebook tests the two GPFA claims, gpfa-variance-inverted-u and gpfa-mean-sigmoid; the
other claims are in `reproduce_sohn_2019.ipynb`. The paper ran GPFA on "high-yield sessions
(N=48 for monkey H ...)" (Results, "The curved manifold hypothesis captures the variance"). This
session has 48 units that pass the unit rule, which suggests it is that session of H; nothing in
the files says so.

| Claim | Tested here |
|---|---|
| gpfa-variance-inverted-u | yes |
| gpfa-mean-sigmoid | yes |

GPFA is the implementation of Yu et al. (2009) in Elephant (`elephant.gpfa.GPFA`), with its
defaults (at most 500 EM iterations) and 20 ms bins. The paper used its own MATLAB code ("custom
MATLAB code", STAR Methods), which is not public.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
import time
import resource
import logging
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import quantities as pq
import neo
from elephant.gpfa import GPFA
from sklearn.model_selection import KFold
import rsg

warnings.filterwarnings("ignore")
logging.getLogger("elephant").setLevel(logging.ERROR)
logging.disable(logging.WARNING)
pd.set_option("display.width", 200)
T0 = time.time()
print("dandiset", rsg.DANDISET_ID, "version", rsg.VERSION)

## Data check

In [ ]:
sess = rsg.load_session()
tr = sess["trials"]
v = tr[tr.valid].copy()
unit_mask, unit_rate = rsg.select_units(sess)
spikes = [s for s, k in zip(sess["spikes"], unit_mask) if k]
print(sess["meta"]["subject"], sess["meta"]["session_start"])
print("trials", len(tr), "| valid (completed, not is_outlier)", len(v))
print("units", len(unit_mask), "| selected (>= 1 spike/s, >= 5 trials in each of 40 conditions)", int(unit_mask.sum()))
print("valid trials per dataset and prior:")
print(v.groupby(["dataset", "prior"]).size().unstack())
d = (v.set_time - v.ready_time) * 1000
print("Set - Ready (ms): min", round(d.min(), 1), "max", round(d.max(), 1),
      "| largest |(Set - Ready) - ts|", round((d - v.ts).abs().max(), 1), "ms")

## GPFA per dataset

Criterion (gpfa-variance-inverted-u): GPFA with 20 ms bins per dataset on single trials from
Ready to Set, both priors, selected units; latent dimensionality chosen by 3-fold cross-validated
log-likelihood from 4 and 8; orthonormalised latents. Each trial is cut at the last whole 20 ms bin
before its Set. Folds are formed over trials with a fixed seed (a standard choice).

The trial-averaged latent trajectory of a prior averages, in each bin, the trials that reach that
bin (attrition, "firing rate at time t was computed from spikes in all trials in which Set
occurred after t", Methods); u is computed from it as in `rsg.encoding_axis`, between the
prior's shortest and longest ts.

In [ ]:
def trial_spiketrains(g):
    out, nb = [], []
    for _, r in g.iterrows():
        n = int(np.floor((r.set_time - r.ready_time) * 1000 / rsg.BIN_MS + 1e-6))
        T = n * rsg.BIN_MS / 1000.0
        out.append([neo.SpikeTrain((s[(s >= r.ready_time) & (s < r.ready_time + T)] - r.ready_time) * pq.s,
                                   t_start=0 * pq.s, t_stop=T * pq.s) for s in spikes])
        nb.append(n)
    return out, np.array(nb)

gp_res = {}
cv_rows = []
for i_ds, ds in enumerate(rsg.DATASETS):
    t1 = time.time()
    g = v[v.dataset == ds]
    sts, nbins = trial_spiketrains(g)
    kf = KFold(3, shuffle=True, random_state=100 + i_ds)
    ll = {4: 0.0, 8: 0.0}
    for k_tr, k_te in kf.split(sts):
        for xd in (4, 8):
            np.random.seed(1000 + i_ds)
            m = GPFA(bin_size=rsg.BIN_MS * pq.ms, x_dim=xd, verbose=False)
            m.fit([sts[j] for j in k_tr])
            ll[xd] += m.score([sts[j] for j in k_te])
    xd = max(ll, key=ll.get)
    np.random.seed(2000 + i_ds)
    m = GPFA(bin_size=rsg.BIN_MS * pq.ms, x_dim=xd, verbose=False)
    lat = m.fit_transform(sts, returned_data=["latent_variable_orth"])
    lat = [np.asarray(x) for x in lat]
    assert all(x.shape[1] == n for x, n in zip(lat, nbins))
    gp_res[ds] = dict(g=g, lat=lat, nbins=nbins, x_dim=xd, n_iter=len(m.fit_info["log_likelihoods"]))
    cv_rows.append(dict(dataset=ds, trials=len(sts), cv_ll_4=ll[4], cv_ll_8=ll[8], chosen=xd,
                        em_iterations=gp_res[ds]["n_iter"], seconds=round(time.time() - t1)))
    print(cv_rows[-1])
cvt = pd.DataFrame(cv_rows)
print(cvt)

In [ ]:
proj_rows = []
for ds in rsg.DATASETS:
    R = gp_res[ds]
    g, lat, nbins = R["g"], R["lat"], R["nbins"]
    for p in rsg.PRIORS:
        idx = np.where(g.prior.values == p)[0]
        nmax = nbins[idx].max()
        S = np.zeros((nmax, R["x_dim"])); N = np.zeros(nmax)
        for j in idx:
            S[:nbins[j]] += lat[j].T; N[:nbins[j]] += 1
        avg = S / N[:, None]
        ends = np.arange(1, nmax + 1) * rsg.BIN_MS
        lo, hi = rsg.TS[p][0], rsg.TS[p][-1]
        u, _ = rsg.encoding_axis(avg, ends, lo, hi)
        for j in idx:
            proj_rows.append(dict(dataset=ds, prior=p, ts=int(g.ts.values[j]), xu=lat[j][:, -1] @ u,
                                  n_bins=int(nbins[j]), n_at_hi=int(N[list(ends).index(hi)])))
proj = pd.DataFrame(proj_rows)
proj["xu_z"] = proj.groupby(["dataset", "prior"]).xu.transform(lambda x: (x - x.mean()) / x.std())
print("single-trial projections:", len(proj), "| xu range", round(proj.xu.min(), 3), "to", round(proj.xu.max(), 3))
print("trials averaged in the last bin of the support (ts_max):",
      proj.groupby(["dataset", "prior"]).n_at_hi.first().to_dict())

### gpfa-variance-inverted-u

> The variance across single trials of the GPFA neural state at Set projected on the encoding
> axis has an inverted-U profile over ts, lower at the extremes of each prior.

Criterion: Xu z-scored over the prior's trials; the variance at each ts; the quadratic
coefficient of a polynomial fit of variance on ts; reproduced when a one-tailed signed-rank test
(coefficient below zero) over the 8 dataset-priors has p < 0.05.

In [ ]:
rows = []
for (ds, p), q in proj.groupby(["dataset", "prior"]):
    var = q.groupby("ts").xu_z.var().reindex(rsg.TS[p])
    coef = np.polyfit(np.array(rsg.TS[p], float), var.values, 2)[0]
    rows.append(dict(dataset=ds, prior=p, var=np.round(var.values, 3), n=q.groupby("ts").size().reindex(rsg.TS[p]).tolist(),
                     quad_coef=coef, inverted_u=coef < 0,
                     extremes_below_middle=(var.values[[0, 4]].mean() < var.values[1:4].mean())))
vq = pd.DataFrame(rows)
print(vq.to_string())
w = stats.wilcoxon(vq.quad_coef, alternative="less")
print("quadratic coefficient < 0 in", int(vq.inverted_u.sum()), "of 8; one-tailed signed-rank p =", round(w.pvalue, 4))
print("mean variance at the extremes below the middle three in", int(vq.extremes_below_middle.sum()), "of 8")

**Result: not reproduced.** The quadratic coefficient is negative in 5 of 8 dataset-priors and
the one-tailed signed-rank test gives p = 0.0742, above 0.05. The mean variance at the two
extreme ts is below that of the middle three in 5 of 8.

### gpfa-mean-sigmoid

> The mean of the single-trial GPFA projections is sigmoidal and has a higher slope for the Short
> prior.

Criterion: with the projections above, not z-scored; the mean Xu at each ts; increments as in
beh-sigmoid-increments, compared by a two-sided signed-rank test over the 8 dataset-priors
(sigmoidal when middle exceeds extreme with p < 0.05); the slope of mean Xu on ts per dataset and
prior (higher for Short in all 4 datasets). Reproduced when both parts hold; partially when one
does.

In [ ]:
rows = []
for (ds, p), q in proj.groupby(["dataset", "prior"]):
    m = q.groupby("ts").xu.mean().reindex(rsg.TS[p]).values
    ext, mid = rsg.increments(m)
    rows.append(dict(dataset=ds, prior=p, mean_xu=np.round(m, 3), extreme=ext, middle=mid,
                     slope_per_s=np.polyfit(np.array(rsg.TS[p], float), m, 1)[0] * 1000))
mq = pd.DataFrame(rows)
print(mq.to_string())
w = stats.wilcoxon(mq.middle, mq.extreme)
print("middle > extreme in", int((mq.middle > mq.extreme).sum()), "of 8; two-sided signed-rank p =", round(w.pvalue, 4))
sl = mq.pivot(index="dataset", columns="prior", values="slope_per_s")
sl["short_higher"] = sl.Short > sl.Long
print(sl.round(4))
print("Short slope higher in", int(sl.short_higher.sum()), "of 4; one-tailed signed-rank p =",
      round(stats.wilcoxon(sl.Short - sl.Long, alternative="greater").pvalue, 4))

**Result: reproduced.** The middle increment of the mean projection exceeds the extreme one in 7
of 8 dataset-priors (two-sided signed-rank p = 0.0156), and the slope of the mean projection on ts
is higher for Short in 4 of 4 datasets (one-tailed signed-rank p = 0.0625, the smallest possible
with 4).

In [ ]:
print("total run time", round(time.time() - T0), "s; peak memory",
      round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2), "GB")

## Summary

| Claim | Outcome |
|---|---|
| gpfa-variance-inverted-u | not reproduced (5 of 8 negative, p = 0.0742) |
| gpfa-mean-sigmoid | reproduced (7 of 8, p = 0.0156; Short higher in 4 of 4) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: GPFA is Elephant's implementation, not the paper's
MATLAB code; cross-validation chose 8 latent dimensions in all 4 datasets. The data are the 48
selected units of the one session in the dandiset, with about 220 to 275 trials per dataset.